In [ ]:
from collections import deque

class SistemManajemenKlinik:
    def __init__(self):
        # --- BAGIAN 3.1: MANAJEMEN PASIEN ---
        # STRUKTUR DATA: Dictionary (Hash Map)
        # JUSTIFIKASI: Memungkinkan pencarian dan pembaruan data pasien berdasarkan Nomor ID
        # dengan kompleksitas waktu O(1)[cite: 20, 22, 52].
        self.pasien_db = {}
        self.counter_id_pasien = 1

        # --- BAGIAN 3.2: MANAJEMEN ANTRIAN ---
        # STRUKTUR DATA: Deque (Double-Ended Queue) dari library bawaan Python
        # JUSTIFIKASI: Operasi penambahan di akhir (append) dan penghapusan di awal (popleft)
        # berjalan dalam waktu O(1), sangat efisien untuk antrian FIFO[cite: 27, 43, 52].
        self.antrian_prioritas = deque()
        self.antrian_reguler = deque()
        self.counter_prioritas = 1
        self.counter_reguler = 1

        # --- BAGIAN 3.3: REKAM MEDIS & RIWAYAT ---
        # STRUKTUR DATA: Dictionary berisikan List yang berfungsi sebagai STACK
        # JUSTIFIKASI: Operasi pembatalan tindakan terakhir (Undo) sangat tepat menggunakan
        # prinsip LIFO (Last In, First Out) dengan fungsi append() dan pop() bernilai O(1)[cite: 31, 52].
        self.riwayat_tindakan = {}

        # --- BAGIAN 3.4: LAPORAN & STATISTIK HARIAN ---
        self.total_pasien_dilayani = 0
        self.log_transaksi = [] # List biasa untuk mencatat riwayat transaksi kronologis [cite: 35]


    # =================================================================================
    # FITUR 3.1: MANAJEMEN PASIEN & OTOMATISASI ANTRIAN (3.2)
    # =================================================================================
    def daftarkan_pasien_baru(self, nama, usia, keluhan, is_darurat=False):
        """Mendaftarkan pasien baru ke database klinik dan memasukkannya ke antrian[cite: 19, 25]."""
        # Pembuatan Nomor ID Pasien Unik Otomatis
        id_pasien = f"ID-{self.counter_id_pasien:03d}"
        self.counter_id_pasien += 1

        # Simpan ke Database Pasien [cite: 19]
        self.pasien_db[id_pasien] = {
            "nama": nama,
            "usia": usia,
            "keluhan": keluhan,
            "is_darurat": is_darurat
        }
        print(f"\n✅ [BERHASIL] Pasien terdaftar dengan Nomor ID Pasien: {id_pasien}")

        # Ketentuan Antrian Prioritas: Pasien darurat ATAU lansia (usia > 60) [cite: 26]
        pasien_antrian = {"id_pasien": id_pasien, "nama": nama, "usia": usia}

        if is_darurat or len(nama) == 0 or usia > 60:
            pasien_antrian["nomor_antrian"] = f"P-{self.counter_prioritas:03d}"
            self.antrian_prioritas.append(pasien_antrian)
            self.counter_prioritas += 1
            print(f"👉 {nama} dimasukkan ke Antrian PRIORITAS ({pasien_antrian['nomor_antrian']}) [cite: 26]")
        else:
            # Antrian Reguler: Pasien umum yang mendaftar secara normal [cite: 26]
            pasien_antrian["nomor_antrian"] = f"R-{self.counter_reguler:03d}"
            self.antrian_reguler.append(pasien_antrian)
            self.counter_reguler += 1
            print(f"👉 {nama} dimasukkan ke Antrian REGULER ({pasien_antrian['nomor_antrian']}) [cite: 26]")

    def cari_data_pasien(self, id_pasien):
        """Mencari data pasien berdasarkan nomor ID dengan cepat O(1)."""
        if id_pasien in self.pasien_db:
            p = self.pasien_db[id_pasien]
            print(f"\n📄 --- DATA PASIEN ({id_pasien}) ---")
            print(f"Nama    : {p['nama']}")
            print(f"Usia    : {p['usia']} tahun")
            print(f"Keluhan : {p['keluhan']}")
            return p
        else:
            print("❌ Error: Nomor ID pasien tidak ditemukan di sistem! [cite: 38]")
            return None

    def tampilkan_seluruh_pasien(self):
        """Menampilkan daftar seluruh pasien yang pernah terdaftar[cite: 21]."""
        print("\n📋 --- DAFTAR SELURUH PASIEN TERDAFTAR ---")
        if not self.pasien_db:
            print("(Belum ada pasien yang terdaftar di database)")
            return
        for id_p, data in self.pasien_db.items():
            status = "Prioritas" if (data['is_darurat'] or data['usia'] > 60) else "Reguler"
            print(f"- ID: {id_p} | Nama: {data['nama']} | Usia: {data['usia']} thn | Status: {status}")

    def perbarui_data_pasien(self, id_pasien, nama_baru, usia_baru, keluhan_baru):
        """Memperbarui data pasien yang sudah ada berdasarkan ID[cite: 22]."""
        if id_pasien in self.pasien_db:
            self.pasien_db[id_pasien]["nama"] = nama_baru
            self.pasien_db[id_pasien]["usia"] = usia_baru
            self.pasien_db[id_pasien]["keluhan"] = keluhan_baru
            print(f"✅ Data pasien dengan ID {id_pasien} berhasil diperbarui! [cite: 22]")
        else:
            print("❌ Error: Gagal memperbarui, ID pasien tidak ditemukan! [cite: 38]")


    # =================================================================================
    # FITUR 3.2: MANAJEMEN ANTRIAN
    # =================================================================================
    def panggil_pasien_berikutnya(self):
        """Memanggil pasien berikutnya dengan mendahulukan Antrian Prioritas[cite: 27]."""
        print("\n📢 [PANGGILAN ANTRIAN]")

        # Mendahulukan Antrian Prioritas [cite: 27]ya5tida
        if self.antrian_prioritas:
            pasien = self.antrian_prioritas.popleft()
            p_data = self.pasien_db[pasien["id_pasien"]]
            keterangan = "Darurat" if p_data["is_darurat"] else f"Lansia ({pasien['usia']} tahun)"
            print(f"🔊 Nomor {pasien['nomor_antrian']} atas nama {pasien['nama']} ({pasien['id_pasien']})")
            print(f"   Kategori: PRIORITAS ({keterangan}) -> Silakan menuju Ruang Pemeriksaan Utama. [cite: 27]")
            return pasien["id_pasien"]

        # Jika prioritas kosong, panggil reguler [cite: 27]
        elif self.antrian_reguler:
            pasien = self.antrian_reguler.popleft()
            print(f"🔊 Nomor {pasien['nomor_antrian']} atas nama {pasien['nama']} ({pasien['id_pasien']})")
            print(f"   Kategori: REGULER -> Silakan menuju Ruang Pemeriksaan. [cite: 27]")
            return pasien["id_pasien"]

        else:
            print("📭 Semua antrian saat ini kosong. [cite: 27]")
            return None

    def tampilkan_kondisi_antrian(self):
        """Menampilkan kondisi antrian saat ini (jumlah dan urutan pasien)[cite: 27]."""
        print("\n============================ STATUS ANTRIAN SAAT INI ============================")

        # 1. Tampilkan Antrian Prioritas [cite: 27]
        print(f"🔴 ANTRIAN PRIORITAS (Total: {len(self.antrian_prioritas)} pasien)")
        if not self.antrian_prioritas:
            print("   (Kosong)")
        else:
            for indeks, pasien in enumerate(self.antrian_prioritas, 1):
                p_data = self.pasien_db[pasien["id_pasien"]]
                kategori = "Darurat" if p_data["is_darurat"] else "Lansia"
                print(f"   {indeks}. [{pasien['nomor_antrian']}] {pasien['nama']} ({pasien['id_pasien']}) - {kategori}")

        print("-" * 80)

        # 2. Tampilkan Antrian Reguler [cite: 27]
        print(f"🔵 ANTRIAN REGULER (Total: {len(self.antrian_reguler)} pasien)")
        if not self.antrian_reguler:
            print("   (Kosong)")
        else:
            for indeks, pasien in enumerate(self.antrian_reguler, 1):
                print(f"   {indeks}. [{pasien['nomor_antrian']}] {pasien['nama']} ({pasien['id_pasien']})")

        print("=================================================================================\n")


    # =================================================================================
    # FITUR 3.3: REKAM MEDIS & RIWAYAT (STACK IMPLEMENTATION)
    # =================================================================================
    def catat_tindakan_medis(self, id_pasien, diagnosa, resep, tindakan, biaya):
        """Mencatat tindakan medis pasien ke dalam Stack[cite: 29]."""
        if id_pasien not in self.pasien_db:
            print("❌ Error: Pasien tidak ditemukan di database! Tindakan medis dibatalkan. [cite: 38]")
            return

        if id_pasien not in self.riwayat_tindakan:
            self.riwayat_tindakan[id_pasien] = []

        # PUSH ke dalam Stack [cite: 29]
        catatan = {"diagnosa": diagnosa, "resep": resep, "tindakan": tindakan, "biaya": biaya}
        self.riwayat_tindakan[id_pasien].append(catatan)

        # Catat transaksi keuangan & statistik [cite: 33, 35]
        self.log_transaksi.append({"id_pasien": id_pasien, "nominal": biaya})
        self.total_pasien_dilayani += 1

        print(f"✅ Tindakan medis untuk {self.pasien_db[id_pasien]['nama']} berhasil disimpan! [cite: 29]")

    def tampilkan_riwayat_terakhir(self, id_pasien):
        """Menampilkan rekam medis paling terakhir (PEEK pada Stack)[cite: 30]."""
        if id_pasien not in self.riwayat_tindakan or not self.riwayat_tindakan[id_pasien]:
            print("⚠️ Belum ada riwayat tindakan medis untuk pasien ini. [cite: 30]")
            return

        # Mengakses elemen paling akhir tanpa menghapusnya (Peek)
        terakhir = self.riwayat_tindakan[id_pasien][-1]
        print(f"\n⚕️ --- RIWAYAT MEDIS TERAKHIR PASIEN ({id_pasien}) --- [cite: 30]")
        print(f"Diagnosa : {terakhir['diagnosa']}")
        print(f"Resep    : {terakhir['resep']}")
        print(f"Tindakan : {terakhir['tindakan']}")

    def batalkan_tindakan_terakhir(self, id_pasien):
        """Membatalkan (Undo) rekaman medis paling terakhir (POP dari Stack)."""
        if id_pasien not in self.riwayat_tindakan or not self.riwayat_tindakan[id_pasien]:
            print("❌ Error: Tidak ada tindakan yang bisa dibatalkan untuk pasien ini! [cite: 31, 38]")
            return

        # POP dari Stack untuk pembatalan LIFO
        dibatalkan = self.riwayat_tindakan[id_pasien].pop()

        # Sinkronisasi: hapus log pembayaran terakhir milik pasien tersebut dari laporan harian
        for i in reversed(range(len(self.log_transaksi))):
            if self.log_transaksi[i]["id_pasien"] == id_pasien:
                self.log_transaksi.pop(i)
                self.total_pasien_dilayani -= 1
                break

        print(f"✅ [UNDO BERHASIL] Tindakan terakhir ({dibatalkan['diagnosa']}) pasien {id_pasien} telah dihapus! ")


    # =================================================================================
    # FITUR 3.4: LAPORAN & STATISTIK HARIAN
    # =================================================================================
    def tampilkan_laporan_harian(self, kriteria_sort):
        """Menampilkan total pelayanan, daftar terurut, dan log transaksi[cite: 32, 33, 34, 35]."""
        print("\n" + "="*40)
        print(" 📊 LAPORAN & STATISTIK HARIAN KLINIK ")
        print("="*40)
        print(f"Total Pasien Selesai Dilayani Hari Ini: {self.total_pasien_dilayani} pasien [cite: 33]")

        # Mengurutkan daftar pasien berdasarkan kriteria [cite: 34]
        print(f"\n--- Daftar Pasien Terdaftar (Urut Berdasarkan {kriteria_sort}) ---")
        if not self.pasien_db:
            print("Belum ada pasien yang terdaftar.")
        else:
            if kriteria_sort == "Nama":
                # Sorting Timsort O(n log n) berdasarkan Alphabet Nama
                pasien_terurut = sorted(self.pasien_db.items(), key=lambda x: x[1]["nama"].lower())
            else:
                # Sorting Timsort O(n log n) berdasarkan ID Pasien
                pasien_terurut = sorted(self.pasien_db.items(), key=lambda x: x[0])

            for id_p, data in pasien_terurut:
                print(f"- ID: {id_p} | Nama: {data['nama']} | Usia: {data['usia']} thn")

        # Riwayat log transaksi pembayaran [cite: 35]
        print("\n--- Log Transaksi Keuangan Hari Ini ---")
        if not self.log_transaksi:
            print("Belum ada transaksi pembayaran masuk. [cite: 35]")
        else:
            total_pemasukan = 0
            for t in self.log_transaksi:
                print(f"- Pasien: {t['id_pasien']} | Nominal: Rp{t['nominal']:,}")
                total_pemasukan += t['nominal']
            print(f"💰 Total Pemasukan Hari Ini: Rp{total_pemasukan:,}")


# =================================================================================
# FITUR 3.5: ANTARMUKA TERMINAL INTERAKTIF (MAIN RUNNER)
# =================================================================================
if __name__ == "__main__":
    klinik = SistemManajemenKlinik()

    while True: # Loop berjalan terus menerus hingga memilih keluar [cite: 39]
        print("\n" + "="*45)
        print("   SISTEM INTEGRASI KLINIK SEHAT BERSAMA   ")
        print("="*45)
        print("[1] Daftarkan Pasien Baru (Otomatis Antrian)")
        print("[2] Panggil Pasien Berikutnya")
        print("[3] Cari Data Pasien Berdasarkan ID")
        print("[4] Perbarui (Update) Data Pasien")
        print("[5] Tampilkan Seluruh Pasien Terdaftar")
        print("[6] Catat Tindakan Medis & Biaya")
        print("[7] Lihat Riwayat Medis Terakhir Pasien")
        print("[8] Batalkan Tindakan Terakhir Pasien (Undo)")
        print("[9] Tampilkan Antrian Saat Ini")
        print("[10] Laporan & Statistik Harian Klinik")
        print("[0] Keluar Aplikasi")
        print("="*45)

        pilihan = input("Pilih nomor menu (0-10): ") # Interaksi menggunakan angka [cite: 37]

        # Validasi Input Angka Menu Utama [cite: 38]
        if pilihan == '1':
            nama = input("Masukkan Nama Pasien: ")
            # Loop validasi input angka usia agar tidak crash [cite: 38]
            while True:
                try:
                    usia = int(input("Masukkan Usia Pasien: "))
                    break
                except ValueError:
                    print("❌ Error: Usia harus berupa angka bulat positif! Silakan coba lagi. [cite: 38]")
            keluhan = input("Keluhan Medis Pasien: ")
            is_darurat_in = input("Apakah pasien kondisi darurat? (ya/tidak): ").strip().lower()
            is_darurat = True if is_darurat_in == 'ya' else False

            klinik.daftarkan_pasien_baru(nama, usia, keluhan, is_darurat)

        elif pilihan == '2':
            klinik.panggil_pasien_berikutnya()

        elif pilihan == '3':
            id_cari = input("Masukkan ID Pasien yang dicari (contoh: ID-001): ").strip().upper()
            klinik.cari_data_pasien(id_cari)

        elif pilihan == '4':
            id_up = input("Masukkan ID Pasien yang ingin diperbarui: ").strip().upper()
            if id_up in klinik.pasien_db:
                nama_n = input("Masukkan Nama Baru: ")
                while True:
                    try:
                        usia_n = int(input("Masukkan Usia Baru: "))
                        break
                    except ValueError:
                        print("❌ Error: Usia harus berupa angka bulat! [cite: 38]")
                keluhan_n = input("Masukkan Keluhan Baru: ")
                klinik.perbarui_data_pasien(id_up, nama_n, usia_n, keluhan_n)
            else:
                print("❌ Error: ID Pasien tidak ditemukan! [cite: 38]")

        elif pilihan == '5':
            klinik.tampilkan_seluruh_pasien()

        elif pilihan == '6':
            id_medis = input("Masukkan ID Pasien yang diperiksa: ").strip().upper()
            if id_medis in klinik.pasien_db:
                diagnosa = input("Masukkan Diagnosa Medis: ")
                resep = input("Masukkan Resep Obat: ")
                tindakan = input("Masukkan Tindakan yang Diberikan: ")
                while True:
                    try:
                        biaya = int(input("Masukkan Total Nominal Biaya Pembayaran: Rp"))
                        break
                    except ValueError:
                        print("❌ Error: Nominal biaya harus berupa angka bulat! [cite: 38]")
                klinik.catat_tindakan_medis(id_medis, diagnosa, resep, tindakan, biaya)
            else:
                print("❌ Error: ID Pasien tidak ditemukan! [cite: 38]")

        elif pilihan == '7':
            id_view = input("Masukkan ID Pasien: ").strip().upper()
            klinik.tampilkan_riwayat_terakhir(id_view)

        elif pilihan == '8':
            id_undo = input("Masukkan ID Pasien yang ingin dibatalkan tindakan terakhirnya: ").strip().upper()
            klinik.batalkan_tindakan_terakhir(id_undo)

        elif pilihan == '9':
            klinik.tampilkan_kondisi_antrian()

        elif pilihan == '10':
            print("\nPilih Kriteria Pengurutan Daftar Pasien:")
            print("[A] Urutkan Alfabetis Nama")
            print("[B] Urutkan Berdasarkan ID Pasien")
            krt = input("Pilih kriteria (A/B): ").strip().upper()

            if krt == 'A':
                klinik.tampilkan_laporan_harian("Nama")
            elif krt == 'B':
                klinik.tampilkan_laporan_harian("ID Pasien")
            else:
                print("❌ Pilihan kriteria tidak valid. Menampilkan urutan default (ID). [cite: 38]")
                klinik.tampilkan_laporan_harian("ID Pasien")

        elif pilihan == '0':
            print("\nTerima kasih! Keluar dari Sistem Manajemen Klinik Sehat Bersama.")
            break # Hentikan program (loop break) [cite: 39]

        else:
            print("❌ Error: Pilihan menu tidak valid! Masukkan angka antara 0 - 10. [cite: 38]")


   SISTEM INTEGRASI KLINIK SEHAT BERSAMA   
[1] Daftarkan Pasien Baru (Otomatis Antrian)
[2] Panggil Pasien Berikutnya
[3] Cari Data Pasien Berdasarkan ID
[4] Perbarui (Update) Data Pasien
[5] Tampilkan Seluruh Pasien Terdaftar
[6] Catat Tindakan Medis & Biaya
[7] Lihat Riwayat Medis Terakhir Pasien
[8] Batalkan Tindakan Terakhir Pasien (Undo)
[9] Tampilkan Antrian Saat Ini
[10] Laporan & Statistik Harian Klinik
[0] Keluar Aplikasi


KeyboardInterrupt: Interrupted by user

In [ ]:
# -*- coding: utf-8 -*-
"""
SISTEM INTEGRASI MANAJEMEN KLINIK SEHAT BERSAMA
================================================
Menggunakan struktur data:
- Dictionary  : Database pasien, O(1) lookup
- Deque       : Antrian prioritas & reguler, O(1) enqueue/dequeue
- List (Stack): Rekam medis LIFO untuk fitur Undo, O(1) push/pop
"""

from collections import deque


# =================================================================================
# KONSTANTA KODE PRIORITAS
# =================================================================================
# P1 = Super Prioritas : Darurat DAN (lansia ATAU keluhan kritis)
# P2 = Prioritas       : Darurat ATAU lansia ATAU keluhan kritis
# R  = Reguler         : Tidak memenuhi kriteria di atas
KELUHAN_KRITIS = [
    "patah tulang", "sesak nafas", "sesak napas", "tidak sadar",
    "pingsan", "serangan jantung", "stroke", "pendarahan",
    "kecelakaan", "luka berat", "kejang"
]

BATAS_USIA_LANSIA = 60
SENTINEL_BATAL = ""  # User tekan Enter kosong = batal


# =================================================================================
# FUNGSI HELPER
# =================================================================================
def input_batal(prompt, tipe=str):
    """
    Input dengan opsi batal. Kembalikan (nilai, True) jika valid,
    atau (None, False) jika user membatalkan (input kosong / ketik 'batal').
    """
    while True:
        raw = input(prompt).strip()
        if raw.lower() in ("", "batal"):
            print("↩️  Operasi dibatalkan, kembali ke menu utama.")
            return None, False
        if tipe == int:
            try:
                return int(raw), True
            except ValueError:
                print("❌ Harus berupa angka bulat. Coba lagi atau tekan Enter untuk batal.")
        else:
            return raw, True


def hitung_kode_prioritas(usia, keluhan, is_darurat):
    """
    Menentukan kode prioritas pasien berdasarkan tiga faktor:
      - is_darurat  : Kondisi darurat (diisi saat pendaftaran)
      - usia        : Lansia jika > 60 tahun
      - keluhan     : Tergolong kritis berdasarkan KELUHAN_KRITIS

    Tingkatan:
      P1 (Super Prioritas) : >=2 faktor terpenuhi
      P2 (Prioritas)       : tepat 1 faktor terpenuhi
      R  (Reguler)         : tidak ada faktor terpenuhi
    """
    keluhan_lower = keluhan.lower()
    faktor_kritis = any(k in keluhan_lower for k in KELUHAN_KRITIS)
    is_lansia = usia > BATAS_USIA_LANSIA

    skor = sum([is_darurat, is_lansia, faktor_kritis])

    if skor >= 2:
        return "P1"
    elif skor == 1:
        return "P2"
    else:
        return "R"


def label_kode(kode):
    """Mengembalikan label deskriptif dari kode prioritas."""
    return {
        "P1": "🔴 SUPER PRIORITAS",
        "P2": "🟡 PRIORITAS",
        "R":  "🔵 REGULER",
    }.get(kode, "TIDAK DIKETAHUI")


# =================================================================================
# KELAS UTAMA
# =================================================================================
class SistemManajemenKlinik:
    def __init__(self):
        # STRUKTUR DATA: Dictionary — pencarian O(1) berdasarkan ID pasien
        self.pasien_db = {}
        self.counter_id_pasien = 1

        # STRUKTUR DATA: Deque — antrian per level prioritas, enqueue/dequeue O(1)
        # Urutan layanan: P1 → P2 → R
        self.antrian_p1 = deque()   # Super Prioritas
        self.antrian_p2 = deque()   # Prioritas
        self.antrian_r  = deque()   # Reguler
        self.counter_p1 = 1
        self.counter_p2 = 1
        self.counter_r  = 1

        # STRUKTUR DATA: Dict of List sebagai Stack — rekam medis LIFO (Undo support)
        self.riwayat_tindakan = {}

        # Log transaksi harian (List kronologis)
        self.total_pasien_dilayani = 0
        self.log_transaksi = []

    # =============================================================================
    # FITUR 1: MANAJEMEN PASIEN
    # =============================================================================
    def daftarkan_pasien_baru(self, nama, usia, keluhan, is_darurat=False):
        """Mendaftarkan pasien baru dan otomatis masuk antrian sesuai kode prioritas."""
        id_pasien = f"ID-{self.counter_id_pasien:03d}"
        self.counter_id_pasien += 1

        kode = hitung_kode_prioritas(usia, keluhan, is_darurat)

        self.pasien_db[id_pasien] = {
            "nama"      : nama,
            "usia"      : usia,
            "keluhan"   : keluhan,
            "is_darurat": is_darurat,
            "kode"      : kode,
        }
        print(f"\n✅ Pasien terdaftar → ID: {id_pasien} | Kode: {label_kode(kode)}")

        pasien_antrian = {"id_pasien": id_pasien, "nama": nama, "usia": usia}

        if kode == "P1":
            pasien_antrian["nomor_antrian"] = f"P1-{self.counter_p1:03d}"
            self.antrian_p1.append(pasien_antrian)
            self.counter_p1 += 1
            alasan = self._alasan_prioritas(usia, keluhan, is_darurat)
            print(f"   → Antrian {pasien_antrian['nomor_antrian']} | Alasan: {alasan}")
        elif kode == "P2":
            pasien_antrian["nomor_antrian"] = f"P2-{self.counter_p2:03d}"
            self.antrian_p2.append(pasien_antrian)
            self.counter_p2 += 1
            alasan = self._alasan_prioritas(usia, keluhan, is_darurat)
            print(f"   → Antrian {pasien_antrian['nomor_antrian']} | Alasan: {alasan}")
        else:
            pasien_antrian["nomor_antrian"] = f"R-{self.counter_r:03d}"
            self.antrian_r.append(pasien_antrian)
            self.counter_r += 1
            print(f"   → Antrian {pasien_antrian['nomor_antrian']}")

    def _alasan_prioritas(self, usia, keluhan, is_darurat):
        """Membuat string alasan mengapa pasien mendapat prioritas tertentu."""
        alasan = []
        if is_darurat:
            alasan.append("kondisi darurat")
        if usia > BATAS_USIA_LANSIA:
            alasan.append(f"lansia ({usia} thn)")
        keluhan_lower = keluhan.lower()
        for k in KELUHAN_KRITIS:
            if k in keluhan_lower:
                alasan.append(f"keluhan kritis ({k})")
                break
        return ", ".join(alasan) if alasan else "-"

    def cari_data_pasien(self, id_pasien):
        """Mencari data pasien berdasarkan ID, O(1)."""
        if id_pasien not in self.pasien_db:
            print("❌ ID pasien tidak ditemukan.")
            return None
        p = self.pasien_db[id_pasien]
        print(f"\n📄 DATA PASIEN ({id_pasien})")
        print(f"   Nama    : {p['nama']}")
        print(f"   Usia    : {p['usia']} tahun")
        print(f"   Keluhan : {p['keluhan']}")
        print(f"   Status  : {label_kode(p['kode'])}")
        return p

    def tampilkan_seluruh_pasien(self):
        """Menampilkan semua pasien yang terdaftar."""
        print("\n📋 DAFTAR SELURUH PASIEN TERDAFTAR")
        if not self.pasien_db:
            print("   (Belum ada pasien terdaftar)")
            return
        for id_p, d in self.pasien_db.items():
            print(f"   {id_p} | {d['nama']:<20} | {d['usia']:>3} thn | {label_kode(d['kode'])}")

    def perbarui_data_pasien(self, id_pasien, nama_baru, usia_baru, keluhan_baru):
        """Memperbarui data pasien dan recalculate kode prioritas."""
        if id_pasien not in self.pasien_db:
            print("❌ ID pasien tidak ditemukan.")
            return
        p = self.pasien_db[id_pasien]
        p["nama"]    = nama_baru
        p["usia"]    = usia_baru
        p["keluhan"] = keluhan_baru
        p["kode"]    = hitung_kode_prioritas(usia_baru, keluhan_baru, p["is_darurat"])
        print(f"✅ Data {id_pasien} diperbarui. Kode prioritas baru: {label_kode(p['kode'])}")

    # =============================================================================
    # FITUR 2: MANAJEMEN ANTRIAN (3 LEVEL)
    # =============================================================================
    def panggil_pasien_berikutnya(self):
        """Memanggil pasien berikutnya: P1 → P2 → R."""
        print("\n📢 PANGGILAN ANTRIAN")
        for antrian, kode in [(self.antrian_p1, "P1"),
                              (self.antrian_p2, "P2"),
                              (self.antrian_r,  "R")]:
            if antrian:
                pasien = antrian.popleft()
                p_data = self.pasien_db[pasien["id_pasien"]]
                print(f"🔊 {pasien['nomor_antrian']} → {pasien['nama']} ({pasien['id_pasien']})")
                print(f"   Status  : {label_kode(kode)}")
                print(f"   Keluhan : {p_data['keluhan']}")
                alasan = self._alasan_prioritas(p_data["usia"], p_data["keluhan"], p_data["is_darurat"])
                if alasan != "-":
                    print(f"   Alasan  : {alasan}")
                print("   → Silakan menuju Ruang Pemeriksaan.")
                return pasien["id_pasien"]

        print("📭 Semua antrian kosong.")
        return None

    def tampilkan_kondisi_antrian(self):
        """Menampilkan semua antrian beserta kode dan alasan prioritas."""
        print("\n" + "="*60)
        print(" STATUS ANTRIAN SAAT INI")
        print("="*60)

        grupan = [
            ("🔴 P1 — SUPER PRIORITAS", self.antrian_p1),
            ("🟡 P2 — PRIORITAS",       self.antrian_p2),
            ("🔵 R  — REGULER",         self.antrian_r),
        ]
        for judul, antrian in grupan:
            print(f"\n{judul} (Total: {len(antrian)} pasien)")
            if not antrian:
                print("   (Kosong)")
            else:
                for i, pasien in enumerate(antrian, 1):
                    p = self.pasien_db[pasien["id_pasien"]]
                    alasan = self._alasan_prioritas(p["usia"], p["keluhan"], p["is_darurat"])
                    print(f"   {i}. [{pasien['nomor_antrian']}] {pasien['nama']} | Alasan: {alasan}")
        print("="*60)

    # =============================================================================
    # FITUR 3: REKAM MEDIS & RIWAYAT (STACK)
    # =============================================================================
    def catat_tindakan_medis(self, id_pasien, diagnosa, resep, tindakan, biaya):
        """Push rekam medis ke Stack pasien dan catat transaksi."""
        if id_pasien not in self.pasien_db:
            print("❌ Pasien tidak ditemukan, tindakan dibatalkan.")
            return
        self.riwayat_tindakan.setdefault(id_pasien, [])
        catatan = {"diagnosa": diagnosa, "resep": resep, "tindakan": tindakan, "biaya": biaya}
        self.riwayat_tindakan[id_pasien].append(catatan)
        self.log_transaksi.append({"id_pasien": id_pasien, "nominal": biaya})
        self.total_pasien_dilayani += 1
        print(f"✅ Tindakan medis untuk {self.pasien_db[id_pasien]['nama']} berhasil disimpan.")

    def tampilkan_riwayat_terakhir(self, id_pasien):
        """Peek — melihat rekam medis terakhir tanpa menghapus."""
        if id_pasien not in self.riwayat_tindakan or not self.riwayat_tindakan[id_pasien]:
            print("⚠️  Belum ada riwayat tindakan untuk pasien ini.")
            return
        terakhir = self.riwayat_tindakan[id_pasien][-1]
        print(f"\n⚕️  RIWAYAT MEDIS TERAKHIR ({id_pasien})")
        print(f"   Diagnosa : {terakhir['diagnosa']}")
        print(f"   Resep    : {terakhir['resep']}")
        print(f"   Tindakan : {terakhir['tindakan']}")
        print(f"   Biaya    : Rp{terakhir['biaya']:,}")

    def batalkan_tindakan_terakhir(self, id_pasien):
        """Pop rekam medis terakhir dari Stack (Undo LIFO)."""
        if id_pasien not in self.riwayat_tindakan or not self.riwayat_tindakan[id_pasien]:
            print("❌ Tidak ada tindakan yang bisa dibatalkan untuk pasien ini.")
            return
        dibatalkan = self.riwayat_tindakan[id_pasien].pop()
        # Hapus log transaksi terakhir milik pasien tersebut
        for i in reversed(range(len(self.log_transaksi))):
            if self.log_transaksi[i]["id_pasien"] == id_pasien:
                self.log_transaksi.pop(i)
                self.total_pasien_dilayani -= 1
                break
        print(f"✅ [UNDO] Tindakan '{dibatalkan['diagnosa']}' untuk {id_pasien} berhasil dihapus.")

    # =============================================================================
    # FITUR 4: LAPORAN & STATISTIK HARIAN
    # =============================================================================
    def tampilkan_laporan_harian(self, kriteria_sort):
        """Laporan total pelayanan, daftar terurut, dan log transaksi."""
        print("\n" + "="*50)
        print("  📊 LAPORAN & STATISTIK HARIAN KLINIK")
        print("="*50)
        print(f"Total Pasien Dilayani : {self.total_pasien_dilayani} pasien")

        print(f"\n--- Daftar Pasien (Urut: {kriteria_sort}) ---")
        if not self.pasien_db:
            print("Belum ada pasien terdaftar.")
        else:
            if kriteria_sort == "Nama":
                pasien_terurut = sorted(self.pasien_db.items(), key=lambda x: x[1]["nama"].lower())
            else:
                pasien_terurut = sorted(self.pasien_db.items(), key=lambda x: x[0])
            for id_p, d in pasien_terurut:
                print(f"   {id_p} | {d['nama']:<20} | {d['usia']:>3} thn | {label_kode(d['kode'])}")

        print("\n--- Log Transaksi Keuangan ---")
        if not self.log_transaksi:
            print("Belum ada transaksi.")
        else:
            total = 0
            for t in self.log_transaksi:
                print(f"   {t['id_pasien']} → Rp{t['nominal']:,}")
                total += t["nominal"]
            print(f"💰 Total Pemasukan Hari Ini: Rp{total:,}")


# =================================================================================
# MAIN — ANTARMUKA TERMINAL INTERAKTIF
# =================================================================================
if __name__ == "__main__":
    klinik = SistemManajemenKlinik()

    while True:
        print("\n" + "="*50)
        print("   SISTEM INTEGRASI KLINIK SEHAT BERSAMA   ")
        print("="*50)
        print(" [1]  Daftarkan Pasien Baru")
        print(" [2]  Panggil Pasien Berikutnya")
        print(" [3]  Cari Data Pasien by ID")
        print(" [4]  Perbarui Data Pasien")
        print(" [5]  Tampilkan Seluruh Pasien")
        print(" [6]  Catat Tindakan Medis & Biaya")
        print(" [7]  Lihat Riwayat Medis Terakhir")
        print(" [8]  Batalkan Tindakan Terakhir (Undo)")
        print(" [9]  Tampilkan Status Antrian")
        print(" [10] Laporan & Statistik Harian")
        print(" [0]  Keluar Aplikasi")
        print("="*50)
        print("  ℹ️  Pada setiap input, tekan Enter kosong / ketik 'batal' untuk membatalkan.")
        print("-"*50)

        pilihan = input("Pilih menu (0-10): ").strip()

        # ------------------------------------------------------------------ MENU 1
        if pilihan == "1":
            print("\n─── DAFTARKAN PASIEN BARU ───")

            nama, ok = input_batal("Nama Pasien         : ")
            if not ok: continue

            usia, ok = input_batal("Usia Pasien (tahun) : ", tipe=int)
            if not ok: continue

            keluhan, ok = input_batal("Keluhan Medis       : ")
            if not ok: continue

            print("Kondisi darurat? Contoh keluhan darurat:", ", ".join(KELUHAN_KRITIS[:5]), "...")
            darurat_raw, ok = input_batal("Darurat? (ya/tidak) : ")
            if not ok: continue
            is_darurat = darurat_raw.lower() == "ya"

            klinik.daftarkan_pasien_baru(nama, usia, keluhan, is_darurat)

        # ------------------------------------------------------------------ MENU 2
        elif pilihan == "2":
            klinik.panggil_pasien_berikutnya()

        # ------------------------------------------------------------------ MENU 3
        elif pilihan == "3":
            print("\n─── CARI DATA PASIEN ───")
            id_cari, ok = input_batal("ID Pasien (cth: ID-001) : ")
            if not ok: continue
            klinik.cari_data_pasien(id_cari.upper())

        # ------------------------------------------------------------------ MENU 4
        elif pilihan == "4":
            print("\n─── PERBARUI DATA PASIEN ───")
            id_up, ok = input_batal("ID Pasien yang diperbarui : ")
            if not ok: continue
            id_up = id_up.upper()
            if id_up not in klinik.pasien_db:
                print("❌ ID tidak ditemukan.")
                continue

            nama_n, ok = input_batal("Nama Baru     : ")
            if not ok: continue

            usia_n, ok = input_batal("Usia Baru     : ", tipe=int)
            if not ok: continue

            keluhan_n, ok = input_batal("Keluhan Baru  : ")
            if not ok: continue

            klinik.perbarui_data_pasien(id_up, nama_n, usia_n, keluhan_n)

        # ------------------------------------------------------------------ MENU 5
        elif pilihan == "5":
            klinik.tampilkan_seluruh_pasien()

        # ------------------------------------------------------------------ MENU 6
        elif pilihan == "6":
            print("\n─── CATAT TINDAKAN MEDIS ───")
            id_medis, ok = input_batal("ID Pasien yang diperiksa : ")
            if not ok: continue
            id_medis = id_medis.upper()
            if id_medis not in klinik.pasien_db:
                print("❌ ID tidak ditemukan.")
                continue

            diagnosa, ok = input_batal("Diagnosa Medis  : ")
            if not ok: continue

            resep, ok = input_batal("Resep Obat      : ")
            if not ok: continue

            tindakan, ok = input_batal("Tindakan        : ")
            if not ok: continue

            biaya, ok = input_batal("Biaya (Rp)      : ", tipe=int)
            if not ok: continue

            klinik.catat_tindakan_medis(id_medis, diagnosa, resep, tindakan, biaya)

        # ------------------------------------------------------------------ MENU 7
        elif pilihan == "7":
            print("\n─── RIWAYAT MEDIS TERAKHIR ───")
            id_view, ok = input_batal("ID Pasien : ")
            if not ok: continue
            klinik.tampilkan_riwayat_terakhir(id_view.upper())

        # ------------------------------------------------------------------ MENU 8
        elif pilihan == "8":
            print("\n─── BATALKAN TINDAKAN TERAKHIR (UNDO) ───")
            id_undo, ok = input_batal("ID Pasien : ")
            if not ok: continue
            klinik.batalkan_tindakan_terakhir(id_undo.upper())

        # ------------------------------------------------------------------ MENU 9
        elif pilihan == "9":
            klinik.tampilkan_kondisi_antrian()

        # ----------------------------------------------------------------- MENU 10
        elif pilihan == "10":
            print("\n─── LAPORAN HARIAN ───")
            print("[A] Urutkan Alfabetis Nama")
            print("[B] Urutkan Berdasarkan ID Pasien")
            krt, ok = input_batal("Pilih kriteria (A/B) atau Enter untuk batal : ")
            if not ok: continue
            krt = krt.upper()
            if krt == "A":
                klinik.tampilkan_laporan_harian("Nama")
            elif krt == "B":
                klinik.tampilkan_laporan_harian("ID Pasien")
            else:
                print("Pilihan tidak valid, tampil urutan default (ID).")
                klinik.tampilkan_laporan_harian("ID Pasien")

        # ------------------------------------------------------------------ MENU 0
        elif pilihan == "0":
            print("\nTerima kasih! Sistem Klinik Sehat Bersama ditutup. 👋")
            break

        else:
            print("❌ Pilihan tidak valid. Masukkan angka 0 - 10.")d


   SISTEM INTEGRASI KLINIK SEHAT BERSAMA   
 [1]  Daftarkan Pasien Baru
 [2]  Panggil Pasien Berikutnya
 [3]  Cari Data Pasien by ID
 [4]  Perbarui Data Pasien
 [5]  Tampilkan Seluruh Pasien
 [6]  Catat Tindakan Medis & Biaya
 [7]  Lihat Riwayat Medis Terakhir
 [8]  Batalkan Tindakan Terakhir (Undo)
 [9]  Tampilkan Status Antrian
 [10] Laporan & Statistik Harian
 [0]  Keluar Aplikasi
  ℹ️  Pada setiap input, tekan Enter kosong / ketik 'batal' untuk membatalkan.
--------------------------------------------------
Pilih menu (0-10): 1

─── DAFTARKAN PASIEN BARU ───
Nama Pasien         : budi
Usia Pasien (tahun) : 50
Keluhan Medis       : patah tulang
Kondisi darurat? Contoh keluhan darurat: patah tulang, sesak nafas, sesak napas, tidak sadar, pingsan ...
Darurat? (ya/tidak) : ya

✅ Pasien terdaftar → ID: ID-001 | Kode: 🔴 SUPER PRIORITAS
   → Antrian P1-001 | Alasan: kondisi darurat, keluhan kritis (patah tulang)

   SISTEM INTEGRASI KLINIK SEHAT BERSAMA   
 [1]  Daftarkan Pasien Baru
 [2